# 01 — Pipeline de datos: extracción, carga y SQL

Este notebook muestra el flujo completo de datos del proyecto:

1. Extracción de series desde las APIs (o modo demo)
2. Carga a la base de datos relacional (SQLite)
3. Consultas SQL con JOIN, GROUP BY y funciones de ventana

> **Nota:** si aún no tienen credenciales, usen `DEMO = True` para trabajar
> con datos sintéticos mientras las tramitan.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from sqlalchemy import text

import config
import carga_db

# Cambiar a False cuando tengan sus credenciales en .env
DEMO = True

## 1. Extraer y cargar las series del catálogo

El catálogo de series se define en `src/config.py`. Este paso descarga todas las series y las guarda en `data/coyuntura.db`.

In [2]:
carga_db.main(demo=DEMO)

[OK] Esquema creado/verificado.
[OK] Catálogo cargado: 6 series.
Extrayendo series...
  [OK] Tipo de cambio observado (CLP/USD): 3278 observaciones
  [OK] Tasa de Política Monetaria (TPM): 3278 observaciones
  [OK] IPC, variación mensual: 199 observaciones
  [OK] Imacec (índice): 199 observaciones
  [OK] IPC Estados Unidos (índice): 199 observaciones
  [OK] Tasa Fed Funds (EE.UU.): 199 observaciones
[OK] Observaciones cargadas/actualizadas: 7352.
Base de datos lista en: /home/claude/kit/kit-inicio-coyuntura/data/coyuntura.db


## 2. Explorar la base de datos con SQL

La gracia de tener los datos en una base relacional es poder consolidarlos con SQL. Veamos las tablas del esquema:

In [3]:
engine = carga_db.obtener_engine()

with engine.connect() as conn:
    tablas = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type IN ('table','view')"), conn)
tablas

,name
0,series
1,temas
2,observaciones
3,vista_monitor


### Consulta con JOIN: catálogo con su clasificación temática

In [4]:
consulta_join = text("""
SELECT s.id_serie, s.nombre, t.tema, s.fuente, s.frecuencia, s.unidad
FROM series s
JOIN temas t ON t.id_tema = s.id_tema
ORDER BY t.tema
""")
with engine.connect() as conn:
    catalogo = pd.read_sql(consulta_join, conn)
catalogo

,id_serie,nombre,tema,fuente,frecuencia,unidad
0,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec (índice),Actividad,BCCH,M,índice 2018=100
1,CPIAUCSL,IPC Estados Unidos (índice),Internacional,FRED,M,índice
2,FEDFUNDS,Tasa Fed Funds (EE.UU.),Internacional,FRED,M,% anual
3,F022.TPM.TIN.D001.NO.Z.D,Tasa de Política Monetaria (TPM),Política monetaria,BCCH,D,% anual
4,F074.IPC.VAR.Z.Z.C.M,"IPC, variación mensual",Precios,BCCH,M,var. % mensual
5,F073.TCO.PRE.Z.D,Tipo de cambio observado (CLP/USD),Sector externo,BCCH,D,CLP/USD


### Consulta con GROUP BY: cobertura de cada serie

In [5]:
consulta_groupby = text("""
SELECT s.nombre,
       MIN(o.fecha) AS primera_obs,
       MAX(o.fecha) AS ultima_obs,
       COUNT(*)     AS n_obs
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
GROUP BY s.nombre
ORDER BY s.nombre
""")
with engine.connect() as conn:
    cobertura = pd.read_sql(consulta_groupby, conn)
cobertura

,nombre,primera_obs,ultima_obs,n_obs
0,IPC Estados Unidos (índice),2010-01-01,2026-07-01,199
1,"IPC, variación mensual",2010-01-01,2026-07-01,199
2,Imacec (índice),2010-01-01,2026-07-01,199
3,Tasa Fed Funds (EE.UU.),2010-01-01,2026-07-01,199
4,Tasa de Política Monetaria (TPM),2014-01-01,2026-07-24,3278
5,Tipo de cambio observado (CLP/USD),2014-01-01,2026-07-24,3278


### Consulta con función de ventana: variación interanual en SQL

La transformación más usada en análisis de coyuntura, calculada directamente en la base con `LAG(..., 12)`:

In [6]:
consulta_ventana = text("""
SELECT o.fecha, s.nombre, o.valor,
       ROUND(100.0 * (o.valor / LAG(o.valor, 12) OVER (PARTITION BY o.id_serie ORDER BY o.fecha) - 1), 2)
           AS var_interanual_pct
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
WHERE s.frecuencia = 'M'
ORDER BY s.nombre, o.fecha
""")
with engine.connect() as conn:
    interanual = pd.read_sql(consulta_ventana, conn, parse_dates=["fecha"])
interanual.dropna().tail(8)

,fecha,nombre,valor,var_interanual_pct
788,2025-12-01,Tasa Fed Funds (EE.UU.),101.840,3.15
789,2026-01-01,Tasa Fed Funds (EE.UU.),102.997,2.74
790,2026-02-01,Tasa Fed Funds (EE.UU.),104.775,2.09
791,2026-03-01,Tasa Fed Funds (EE.UU.),106.036,1.22
792,2026-04-01,Tasa Fed Funds (EE.UU.),107.332,2.25
793,2026-05-01,Tasa Fed Funds (EE.UU.),106.974,2.66
794,2026-06-01,Tasa Fed Funds (EE.UU.),105.324,0.92
795,2026-07-01,Tasa Fed Funds (EE.UU.),103.404,0.91


## 3. Siguiente paso

Con los datos en la base, el análisis continúa en `02_analisis_ejemplo.ipynb`.

**Tareas del equipo:**
- Reemplazar el catálogo de `config.py` por sus propias series (≥ 12)
- Escribir sus propias consultas en `sql/` adaptadas a su variable objetivo